In [14]:
import requests
from pathlib import Path
import pandas as pd

from geopy.geocoders import Nominatim

dir = Path('C:/Users/Marcus/Documents/DSAI/Azure_WBS/silver/')
datelist = []

for i in ["flow/", "weight/", "humidity/", "temperature/"]:
    parquet_files = list(dir.joinpath(i).glob('*.parquet'))

    if not parquet_files:
        raise FileNotFoundError(f'No parquet files found in {dir}/{i}')

    newest_file = max(parquet_files, key=lambda p: p.stat().st_mtime)

    print(f"Reading {i}....", end="")
    df = pd.read_parquet(newest_file)
    mindate = df["timestamp"].min()
    maxdate = df["timestamp"].max() 
    print(f"\tMin:{mindate}, Max::{maxdate}")
    datelist.append(mindate)
    datelist.append(maxdate)

mindate=min(datelist)
maxdate=max(datelist)
print(f"Overall: \tMin:{mindate}, Max::{maxdate}")



geolocator = Nominatim(user_agent="geoapi")
location_name = newest_file.stem.split('__')[0]
location = geolocator.geocode(location_name)

print(f"Found {location_name} at {location.latitude}, {location.longitude}")

# Construct API request URL
url = f"https://api.brightsky.dev/weather?lat={location.latitude}&lon={location.longitude}&date={mindate}&last_date={maxdate}"
print(url)
# Send the request
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    weather_data = response.json()  # Convert response to JSON format
    weather_df = pd.DataFrame(weather_data.get('weather', []))
    print(weather_df.head(20))
     # Preview weather rows
else:
    print(f"Error: Unable to retrieve data ({response.status_code})")


Reading flow/....	Min:2017-01-01 13:15:00+00:00, Max::2019-05-31 12:15:00+00:00
Reading weight/....	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:00:00+00:00
Reading humidity/....	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:00:00+00:00
Reading temperature/....	Min:2017-01-01 13:10:00+00:00, Max::2019-05-31 12:15:00+00:00
Overall: 	Min:2017-01-01 12:00:00+00:00, Max::2019-05-31 12:15:00+00:00
Found schwartau at 54.0099465, 10.6754006
https://api.brightsky.dev/weather?lat=54.0099465&lon=10.6754006&date=2017-01-01 12:00:00+00:00&last_date=2019-05-31 12:15:00+00:00
                    timestamp  source_id  precipitation  pressure_msl  \
0   2017-01-01T12:00:00+00:00     286541            0.0        1012.6   
1   2017-01-01T13:00:00+00:00     286541            0.0        1012.0   
2   2017-01-01T14:00:00+00:00     286541            0.0        1012.5   
3   2017-01-01T15:00:00+00:00     286541            0.0           NaN   
4   2017-01-01T16:00:00+00:00     286541            0.0  

In [22]:
# 1) Columns that contain at least one missing value
missing_cols = weather_df.columns[weather_df.isna().any()].tolist()
print("Columns with missing values:", missing_cols)

# 2) Missing count per column (only columns with missing values)
missing_count = weather_df.isna().sum()
print(missing_count[missing_count > 0].sort_values(ascending=False))


Columns with missing values: ['pressure_msl', 'sunshine', 'temperature', 'wind_direction', 'wind_speed', 'cloud_cover', 'dew_point', 'relative_humidity', 'visibility', 'wind_gust_direction', 'wind_gust_speed', 'condition', 'precipitation_probability', 'precipitation_probability_6h', 'solar', 'fallback_source_ids']
precipitation_probability       21119
precipitation_probability_6h    21119
sunshine                         5423
solar                            5205
visibility                       4744
cloud_cover                      4737
pressure_msl                     4736
temperature                      4736
wind_direction                   4736
wind_speed                       4736
dew_point                        4736
relative_humidity                4736
wind_gust_direction              4736
wind_gust_speed                  4736
condition                         510
fallback_source_ids               219
dtype: int64


In [3]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="geoapi")
location = geolocator.geocode("schwartau")

print(location.latitude, location.longitude)


54.0099465 10.6754006
